# Composing a Custom Module

A `dspy.Module` lets you wire multiple signatures into a single pipeline.  
Here we build a two-step module: classify the domain of a question, then rewrite it into a cleaner search query.

In [1]:
import numpy
import dspy
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434')
dspy.configure(lm=lm)

## Step 1 — Define the Signatures

In [2]:
class DomainClassifier(dspy.Signature):
    """Classify the domain of the question."""

    question: str = dspy.InputField(desc="A user's question")
    domain: Literal["science", "history", "sports"] = dspy.OutputField(desc="The domain the question belongs to")


class QueryRewriter(dspy.Signature):
    """Rewrite the question into a precise search query suited to its domain."""

    question: str = dspy.InputField(desc="The original user question")
    domain: str = dspy.InputField(desc="The domain the question belongs to")
    search_query: str = dspy.OutputField(desc="A concise, search-engine-friendly version of the question")

## Step 2 — Build the Module

- Declare sub-modules in `__init__`
- Wire them together in `forward()`
- The output of step 1 feeds into step 2

In [3]:
class QueryUnderstanding(dspy.Module):
    def __init__(self):
        self.classify = dspy.Predict(DomainClassifier)
        self.rewrite = dspy.ChainOfThought(QueryRewriter)

    def forward(self, question):
        classification = self.classify(question=question)
        rewrite = self.rewrite(question=question, domain=classification.domain)
        return dspy.Prediction(
            domain=classification.domain,
            search_query=rewrite.search_query,
            reasoning=rewrite.reasoning
        )

## Step 3 — Run it

In [ ]:
pipeline = QueryUnderstanding()

questions = [
    "who scored the most goals in the world cup",
    "why did the roman empire fall",
    "how does a black hole form",
]

for q in questions:
    result = pipeline(question=q)
    print(f"Q:      {q}")
    print(f"Domain: {result.domain}")
    print(f"Rewrite: {result.search_query}")
    print()

Q:      who scored the most goals in the world cup
Domain: sports
Rewrite:world cup goals scored

Q:      why did the roman empire fall
Domain: history
Rewrite:rome decline and fall

Q:      how does a black hole form
Domain: science
Rewrite:black hole formation process



In [5]:
dspy.inspect_history(n=1)





[2026-08-20T14:27:51.767274]

System message:

Your input fields are:
1. `question` (str): The original user question
2. `domain` (str): The domain the question belongs to
Your output fields are:
1. `reasoning` (str): 
2. `search_query` (str): A concise, search-engine-friendly version of the question
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## domain ## ]]
{domain}

[[ ## reasoning ## ]]
{reasoning}

[[ ## search_query ## ]]
{search_query}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Rewrite the question into a precise search query suited to its domain.


User message:

[[ ## question ## ]]
how does a black hole form

[[ ## domain ## ]]
science

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## search_query ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reas